In [ ]:


%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, HTML

# استيراد مكتبات إصلاح الخط العربي
import arabic_reshaper
from bidi.algorithm import get_display

def run_semtos_simulation(solar_prod, battery_soc, grid_status, cum_cons, current_demand, scenario_title="فحص سيناريو مخصص"):
    """
    تابع شامل لـ SEMTOS محصن تماماً بكتل try-except لحماية النظام من الانهيار عند الحواف الحادة،
    ويعرض المدخلات والمخرجات على شكل جداول HTML خارقة، ويرسم المنحنيات بدقة مشبوكة بالعنوان.
    """
    # 1. تثبيت النمط المظلم للمخططات
    plt.style.use('dark_background')
    energy_sim = ctrl.ControlSystemSimulation(energy_ctrl_system)

    energy_sim.input['battery_soc'] = battery_soc
    energy_sim.input['solar_production'] = solar_prod
    energy_sim.input['grid_status'] = grid_status
    energy_sim.input['cum_consumption'] = cum_cons
    energy_sim.input['current_demand'] = current_demand

    energy_sim.compute()

    # 4. جلب المخرجات الرقمية الخمسة مع الحماية الكاملة وصياغة التقارير ديناميكياً

    # --- أ. الاعتماد على الشبكة ---
    try:
        grid_val = energy_sim.output['grid_dependency']
        if grid_val <= 30:
            grid_report = f"اعتماد منخفض جداً على الشبكة العامة (بنسبة {grid_val:.2f}%) - المنظومة تعتمد على المصادر الذاتية."
        elif 30 < grid_val <= 70:
            grid_report = f"اعتماد متوسط ومدروس على الشبكة (بنسبة {grid_val:.2f}%) - موازنة الأحمال مستقرة."
        else:
            grid_report = f"اعتماد مرتفع وحرج على الشبكة العامة (بنسبة {grid_val:.2f}%) - نتيجة نقص التوليد الذاتي أو زيادة الأحمال."
    except KeyError:
        grid_val = None
        grid_report = "⚠️ حافة حرجة! يتعذر حساب الاعتماد على الشبكة، يتم تشغيل النظام بأقصى درجات الحيطة."

    # --- ب. أولوية شحن البطارية ---
    try:
        charge_priority_val = energy_sim.output['battery_charging_priority']
        if charge_priority_val <= 20:
            charge_priority_report = f"لا حاجة لشحن البطارية حالياً (المستوى: {charge_priority_val:.2f}%)."
        elif charge_priority_val <= 40:
            charge_priority_report = f"أولوية شحن منخفضة (المستوى: {charge_priority_val:.2f}%)."
        elif charge_priority_val <= 60:
            charge_priority_report = f"أولوية شحن متوسطة (المستوى: {charge_priority_val:.2f}%)."
        elif charge_priority_val <= 80:
            charge_priority_report = f"أولوية شحن مرتفعة (المستوى: {charge_priority_val:.2f}%)."
        else:
            charge_priority_report = f"شحن البطارية أولوية قصوى حالياً (المستوى: {charge_priority_val:.2f}%)."
    except KeyError:
        charge_priority_val = None
        charge_priority_report = "لم يتم توليد قرار واضح للشحن."

    # --- ج. إجراء سلوك البطارية ---
    try:
        batt_val = energy_sim.output['battery_action']
        if batt_val <= 45:
            batt_report = f"تفريغ آمن (Safe Discharge بنسبة {batt_val:.2f}%) للمساعدة في تغذية الأحمال وتوفير المال."
        elif 45 < batt_val <= 65:
            batt_report = f"وضعية الاستقرار والمحافظة على المخزون الحالي بدون تفريغ حاد (Standby عند {batt_val:.2f}%)."
        else:
            batt_report = f"شحن مكثف وضخ طاقة للبطارية (بقوة {batt_val:.2f}%) لاستغلال الوفرة وتأمين فترات التقنين القادمة."
    except KeyError:
        batt_val = None
        batt_report = "⚠️ حافة حرجة! المنظومة تفعل وضعية الاستقرار الاحتياطي (Standby) للمحافظة على الخلايا الكيميائية."

    # --- د. التحكم بالأحمال المنزلية ---
    try:
        load_val = energy_sim.output['load_control']
        if load_val <= 45:
            load_report = f"الترشيد الصارم والذكي (Eco Mode بنسبة {load_val:.2f}%) - يسمح بالإنارة والشواحن والبراد فقط."
        elif 45 < load_val <= 75:
            load_report = f"التحكم الذكي المتوسط (Smart Limit بنسبة {load_val:.2f}%) - مسموح بأجهزة متوسطة وتأجيل الثقيلة."
        else:
            load_report = f"الرفاهية الكاملة وتلبية الطلب (Full Power بنسبة {load_val:.2f}%) - مسموح بتشغيل المكيفات والسخانات."
    except KeyError:
        load_val = None
        load_report = "⚠️ حافة حرجة! تم تفعيل نمط الترشيد الصارم (Eco Mode) حركياً لحين استقرار المعطيات الحادة."

    # --- هـ. التحكم بالفائض الشمسي ---
    try:
        solar_val = energy_sim.output['solar_curtailment']
        if solar_prod < 20:
            solar_report = "لا يوجد فائض يذكر نظراً لغياب الأشعة الشمسية الحالية أو ضعف التوليد الشديد."
        else:
            if solar_val <= 50:
                solar_report = f"توجيه كامل الإنتاج الشمسي للاستهلاك المنزلي المباشر الفوري (Direct Use بنسبة {solar_val:.2f}%)."
            elif 50 < solar_val <= 80:
                solar_report = f"يوجد فائض ممتاز من الألواح يتم تحويله ذكياً الآن لحقن البطاريات وتخزينه (Storage بنسبة {solar_val:.2f}%)."
            else:
                solar_report = f"فائض حرج جداً (Dump بنسبة {solar_val:.2f}%) المنظومة مكتفية والبطارية ممتلئة! يتم تصريفه بأمان."
    except KeyError:
        solar_val = None
        solar_report = "⚠️ حافة حرجة! يتم توجيه الطاقة الشمسية المتاحة للاستهلاك المباشر دون تفريغ أو كبح إضافي."

    # --- 🏗️ بناء التنسيق البصري الخارق يدوياً عن طريق HTML ---
    print("\n" + "="*85)
    print(f" 🤖 لوحة تحكم ونظام تقارير SEMTOS الذكي المحمي ")
    print("="*85)

    title_html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; margin-bottom: 10px;">
        <h2 style="color: #ffaa00; margin-bottom: 5px; border-bottom: 2px solid #ffaa00; padding-bottom: 5px;">🎬 سيناريو الفحص: {scenario_title}</h2>
    </div>
    """
    display(HTML(title_html))

    # أ. جدول المدخلات الحالي
    inputs_table_html = f"""
    <h3 style="color: #00ff66; font-family: 'Segoe UI', sans-serif; margin-top: 15px; margin-bottom: 8px;">📥 جدول المدخلات الحالية (System Inputs)</h3>
    <table style="background-color: #121212; border-collapse: collapse; width: 100%; border: 1px solid #444; font-family: 'Segoe UI', sans-serif; margin-bottom: 25px;">
        <thead>
            <tr style="background-color: #222222;">
                <th style="color: #00ff66; padding: 10px; border: 1px solid #444; text-align: center; width: 25%;">المدخل (Arabic)</th>
                <th style="color: #00ff66; padding: 10px; border: 1px solid #444; text-align: center; width: 25%;">Input (English)</th>
                <th style="color: #00ff66; padding: 10px; border: 1px solid #444; text-align: center; width: 20%;">القيمة الممررة (Value)</th>
                <th style="color: #00ff66; padding: 10px; border: 1px solid #444; text-align: center; width: 30%;">الوحدة / الحالة</th>
            </tr>
        </thead>
        <tbody>
            <tr style="background-color: #1a1a1a;">
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">☀️ إنتاج الطاقة الشمسية</td>
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: left;">Solar Production</td>
                <td style="padding: 10px; color: #ffaa00; border: 1px solid #333; text-align: center; font-weight: bold; font-size: 14px;">{solar_prod}%</td>
                <td style="padding: 10px; color: #e0e0e0; border: 1px solid #333; text-align: right;">نسبة توليد الألواح الحالية من القدرة القصوى</td>
            </tr>
            <tr style="background-color: #121212;">
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">🔋 مستوى شحن البطارية</td>
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: left;">Battery SoC</td>
                <td style="padding: 10px; color: #00ff66; border: 1px solid #333; text-align: center; font-weight: bold; font-size: 14px;">{battery_soc}%</td>
                <td style="padding: 10px; color: #e0e0e0; border: 1px solid #333; text-align: right;">حالة شحن بنك البطاريات المتوفر (State of Charge)</td>
            </tr>
            <tr style="background-color: #1a1a1a;">
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">🔌 وضع شبكة الدولة</td>
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: left;">Grid Status</td>
                <td style="padding: 10px; color: #33a1ff; border: 1px solid #333; text-align: center; font-weight: bold; font-size: 14px;">{grid_status}%</td>
                <td style="padding: 10px; color: #e0e0e0; border: 1px solid #333; text-align: right;">استقرار وتوفر تيار الشبكة العامة (الكهرباء النظامية)</td>
            </tr>
            <tr style="background-color: #121212;">
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">📉 استهلاك العداد التراكمي</td>
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: left;">Cumulative Consumption</td>
                <td style="padding: 10px; color: #ff3366; border: 1px solid #333; text-align: center; font-weight: bold; font-size: 14px;">{cum_cons} ك.و.س</td>
                <td style="padding: 10px; color: #e0e0e0; border: 1px solid #333; text-align: right;">الطاقة المستهلكة خلال الدورة الحالية (مؤشر خطر الـ 500)</td>
            </tr>
            <tr style="background-color: #1a1a1a;">
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">🏠 حمل المنزل الحالي</td>
                <td style="padding: 10px; color: #ffffff; border: 1px solid #333; text-align: left;">Current Demand</td>
                <td style="padding: 10px; color: #cc66ff; border: 1px solid #333; text-align: center; font-weight: bold; font-size: 14px;">{current_demand}%</td>
                <td style="padding: 10px; color: #e0e0e0; border: 1px solid #333; text-align: right;">مجموع سحب الأجهزة والإنارة اللحظي داخل المنزل</td>
            </tr>
        </tbody>
    </table>
    """
    display(HTML(inputs_table_html))

    # تجهيز النصوص الرقمية للجدول لمنع طباعة السلسلة الفارغة
    g_val_str = f"{grid_val:.2f}%" if grid_val is not None else "DEFAULT (0%)"
    cp_val_str = f"{charge_priority_val:.2f}%" if charge_priority_val is not None else "DEFAULT (0%)"
    b_val_str = f"{batt_val:.2f}%" if batt_val is not None else "DEFAULT (50%)"
    l_val_str = f"{load_val:.2f}%" if load_val is not None else "DEFAULT (30%)"
    s_val_str = f"{solar_val:.2f}%" if solar_val is not None else "DEFAULT (0%)"

    # ب. جدول المخرجات الذكي
    outputs_table_html = f"""
    <h3 style="color: #00ffff; font-family: 'Segoe UI', sans-serif; margin-top: 10px; margin-bottom: 8px;">📤 جدول المخرجات والقرارات الأكاديمية (Outputs & Decisions)</h3>
    <table style="background-color: #121212; border-collapse: collapse; width: 100%; border: 1px solid #444; font-family: 'Segoe UI', sans-serif;">
        <thead>
            <tr style="background-color: #2d2d2d;">
                <th style="color: #00ffff; padding: 12px; border: 1px solid #444; text-align: center; width: 25%;">الخرج (Arabic)</th>
                <th style="color: #00ffff; padding: 12px; border: 1px solid #444; text-align: center; width: 25%;">Output (English)</th>
                <th style="color: #00ffff; padding: 12px; border: 1px solid #444; text-align: center; width: 15%;">النسبة الحادة (Value)</th>
                <th style="color: #00ffff; padding: 12px; border: 1px solid #444; text-align: center; width: 35%;">التقرير الذكي ونوع القرار (System Action Report)</th>
            </tr>
        </thead>
        <tbody>
            <tr style="background-color: #1a1a1a;">
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">الاعتماد على الشبكة</td>
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: left;">Grid Dependency</td>
                <td style="padding: 12px; color: #00ffff; border: 1px solid #333; text-align: center; font-weight: bold;">{g_val_str}</td>
                <td style="padding: 12px; color: #e0e0e0; border: 1px solid #333; text-align: right;">{grid_report}</td>
            </tr>
            <tr style="background-color: #121212;">
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">أولوية شحن البطارية</td>
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: left;">Battery Charging Priority</td>
                <td style="padding: 12px; color: #ffaa00; border: 1px solid #333; text-align: center; font-weight: bold;">{cp_val_str}</td>
                <td style="padding: 12px; color: #e0e0e0; border: 1px solid #333; text-align: right;">{charge_priority_report}</td>
            </tr>
            <tr style="background-color: #1a1a1a;">
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">إجراء سلوك البطارية</td>
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: left;">Battery Action</td>
                <td style="padding: 12px; color: #ff3333; border: 1px solid #333; text-align: center; font-weight: bold;">{b_val_str}</td>
                <td style="padding: 12px; color: #e0e0e0; border: 1px solid #333; text-align: right;">{batt_report}</td>
            </tr>
            <tr style="background-color: #121212;">
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">التحكم بالأحمال المنزلية</td>
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: left;">Load Control</td>
                <td style="padding: 12px; color: #33ff33; border: 1px solid #333; text-align: center; font-weight: bold;">{l_val_str}</td>
                <td style="padding: 12px; color: #e0e0e0; border: 1px solid #333; text-align: right;">{load_report}</td>
            </tr>
            <tr style="background-color: #1a1a1a;">
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: right; font-weight: bold;">التحكم بالفائض الشمسي</td>
                <td style="padding: 12px; color: #ffffff; border: 1px solid #333; text-align: left;">Solar Curtailment</td>
                <td style="padding: 12px; color: #ffff33; border: 1px solid #333; text-align: center; font-weight: bold;">{s_val_str}</td>
                <td style="padding: 12px; color: #e0e0e0; border: 1px solid #333; text-align: right;">{solar_report}</td>
            </tr>
        </tbody>
    </table>
    """
    display(HTML(outputs_table_html))
    print("\n" + "-"*85 + "\n")

    # 6. رسم المخرجات الخمسة بالكامل يدوياً
    fig, axs = plt.subplots(5, 1, figsize=(10, 22))

    # إصلاح العنوان العربي (إزالة الإيموجي ومعالجة النص ليظهر صحيحاً)
    cleaned_title = f"System Scenario: {scenario_title}"
    reshaped_text = arabic_reshaper.reshape(cleaned_title)
    bidi_title = get_display(reshaped_text)

    fig.suptitle(bidi_title, fontsize=14, color='#ffaa00', weight='bold', y=0.99)

    # المخطط 1: الاعتماد على الشبكة
    for label in grid_dependency.terms:
        axs[0].plot(grid_dependency.universe, grid_dependency[label].mf, label=label, linewidth=2)
    if grid_val is not None:
        axs[0].axvline(x=grid_val, color='#00ffff', linestyle='--', linewidth=3, label=f'Decision: {grid_val:.2f}%')
    axs[0].set_title("1. Grid Dependency Output Spectrum", fontsize=11, color='white')
    axs[0].set_ylabel("Membership", color='white')
    axs[0].legend(loc='upper right')
    axs[0].grid(True, alpha=0.2)

    # المخطط 2: أولوية شحن البطارية
    for label in battery_charging_priority.terms:
        axs[1].plot(battery_charging_priority.universe, battery_charging_priority[label].mf, label=label, linewidth=2)
    if charge_priority_val is not None:
        axs[1].axvline(x=charge_priority_val, color='#ffaa00', linestyle='--', linewidth=3, label=f'Decision: {charge_priority_val:.2f}%')
    axs[1].set_title("2. Battery Charging Priority Output Spectrum", fontsize=11, color='white')
    axs[1].set_ylabel("Membership", color='white')
    axs[1].legend(loc='upper right')
    axs[1].grid(True, alpha=0.2)

    # المخطط 3: سلوك وإجراء البطارية
    for label in battery_action.terms:
        axs[2].plot(battery_action.universe, battery_action[label].mf, label=label, linewidth=2)
    if batt_val is not None:
        axs[2].axvline(x=batt_val, color='#ff3333', linestyle='--', linewidth=3, label=f'Decision: {batt_val:.2f}%')
    axs[2].set_title("3. Battery Action Output Spectrum", fontsize=11, color='white')
    axs[2].set_ylabel("Membership", color='white')
    axs[2].legend(loc='upper right')
    axs[2].grid(True, alpha=0.2)

    # المخطط 4: وضعية التحكم بالأحمال
    for label in load_control.terms:
        axs[3].plot(load_control.universe, load_control[label].mf, label=label, linewidth=2)
    if load_val is not None:
        axs[3].axvline(x=load_val, color='#33ff33', linestyle='--', linewidth=3, label=f'Decision: {load_val:.2f}%')
    axs[3].set_title("4. Load Control Output Spectrum", fontsize=11, color='white')
    axs[3].set_ylabel("Membership", color='white')
    axs[3].legend(loc='upper right')
    axs[3].grid(True, alpha=0.2)

    # المخطط 5: التحكم بفائض الطاقة الشمسية
    for label in solar_curtailment.terms:
        axs[4].plot(solar_curtailment.universe, solar_curtailment[label].mf, label=label, linewidth=2)
    if solar_val is not None:
        axs[4].axvline(x=solar_val, color='#ffff33', linestyle='--', linewidth=3, label=f'Decision: {solar_val:.2f}%')
    axs[4].set_title("5. Solar Curtailment Output Spectrum", fontsize=11, color='white')
    axs[4].set_xlabel("Percentage (%)", color='white')
    axs[4].set_ylabel("Membership", color='white')
    axs[4].legend(loc='upper right')
    axs[4].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
# 1. تحديد قيم عينات الفحص للممررات الخمسة
solar_prod_input     = 85   # إنتاج طاقة شمسية مرتفع بنسبة 85%
battery_soc_input    = 30   # مستوى شحن البطارية الحالي 30%
grid_status_input    = 100  # شبكة الدولة مستقرة ومتوفرة بالكامل 100%
cum_cons_input       = 420  # الاستهلاك التراكمي للعداد (420 كيلو واط ساعي)
current_demand_input = 70   # حمل المنزل اللحظي الحالي 70%

# 2. استدعاء الدالة مع تمرير المتغيرات وتسمية السيناريو ليظهر في جداول HTML والرسومات البيانية
run_semtos_simulation(
    solar_prod=solar_prod_input,
    battery_soc=battery_soc_input,
    grid_status=grid_status_input,
    cum_cons=cum_cons_input,
    current_demand=current_demand_input,
    scenario_title="سيناريو شبك التوليد الأقصى مع استقرار الشبكة العامة"
)